# Validation du preprocessing sur une grille céleste commune

Ce notebook **ne modifie pas l’entraînement ni ses caches**. Il traite le catalogue
LoTSS-DR2/PSZ2 complet, indépendamment de la présence des modalités.

La grille est centrée sur les RA/Dec du catalogue (ICRS/J2000), nord en haut et est
à gauche. Son côté nominal dans le plan tangent est `4 R500` converti avec Planck18.
Si seul M500 et z sont disponibles, R500 est dérivé de `500 × rho_crit(z)`.
Sinon, le champ médian de tous les amas physiquement complets est utilisé.
**Pour un futur entraînement : calculer ce fallback sur le train de chaque fold,
ou utiliser une référence externe fixée à l’avance.**

Le catalogue FITS officiel fournit les unités : RA/Dec en degrés, R500 en kpc,
M500 en 10¹⁴ M☉. Le CSV historique local ne contient pas les coordonnées.
Une copie du catalogue officiel est téléchargée une seule fois dans `inputs/`.
Source : https://lofar-surveys.org/planck_dr2.html (Botteon et al., Paper I).
Aucun centre FITS ne remplace silencieusement une coordonnée manquante.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from astropy.cosmology import Planck18

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "clusterprep/src/clusterprep").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "clusterprep/src"))
from clusterprep.common_wcs_validation import (
    ValidationConfig, fetch_catalog, read_catalog, fingerprint, run_validation,
    median_field, CATALOG_URL,
)

OUTPUT_SIZE = 128
FOV_R500_FACTOR = 4.0
COSMOLOGY = Planck18
PROJECTION = "TAN"
CLUSTER_IDS = None  # tous ; ou "PSZ2G073.97-27.82" ; ou liste d’identifiants
SAVE_FIGURES = True
SAVE_PROCESSED = True
OVERWRITE = False  # True force le recalcul ; sinon réutilise seulement les sorties valides
VIEW = "both"  # "before", "after", "both"
LEGACY_BANDS = ("g", "r", "z")  # bandes individuelles, pas de RGB implicite
COVERAGE_WARNING = 0.8
FIGURE_DPI = 180
OUTPUT_DIR = PROJECT_ROOT / "outputs/common_wcs_validation"
FITS_ROOT = PROJECT_ROOT / "scratch/data/PSZ2/fits"
CATALOG_PATH = OUTPUT_DIR / "inputs/lofar_psz2_catalog.fits"


## Catalogue et médiane de validation

Un catalogue local alternatif doit respecter le schéma `Name, RA, DEC, z, R500,
M500` avec des unités explicites dans FITS ou ECSV. Les valeurs non finies ou
physiques non positives sont ignorées. Les coordonnées invalides donnent une
ligne d’erreur par amas. Un ensemble sans champ physique valide provoque un arrêt.
La sélection d’un seul amas ne change pas la population servant à la médiane.


In [ ]:
fetch_catalog(CATALOG_PATH)
records = read_catalog(CATALOG_PATH)
config = ValidationConfig(
    output_size=OUTPUT_SIZE, fov_r500_factor=FOV_R500_FACTOR,
    projection=PROJECTION, legacy_bands=LEGACY_BANDS, view=VIEW,
    save_figures=SAVE_FIGURES, save_processed=SAVE_PROCESSED,
    dpi=FIGURE_DPI, coverage_warning=COVERAGE_WARNING,
)
print(f"Catalogue : {len(records)} amas ; cosmologie : {COSMOLOGY.name}")
print("La médiane est calculée une fois par run, puis enregistrée dans configuration.json.")


## Reprojection, sauvegarde et reprise

Chaque modalité est lue indépendamment et reprojetée directement sur la grille
commune avec `reproject_interp(order="bilinear")`, sans downsampling dépendant de
la radio. Les masques marquent les valeurs finies couvertes ; les valeurs nulles
physiques restent valides et les régions sans données restent NaN dans les FITS.
Les bandes Legacy absentes sont signalées, sans inventer de canaux.

**Limites :** interpolation bilinéaire, sans conservation du flux intégré en
Jy/beam, sans égalisation des beams et sans conversion photométrique. Elle peut
sous-échantillonner les sources compactes ; ce notebook valide la géométrie,
pas une mesure photométrique finale. Les métadonnées beam conservées décrivent
le beam d’origine, pas un nouveau beam effectif.

La normalisation appelle directement `clusterprep.preprocessing.normalize` avec
P30/P99 et alpha=10, **par image et par bande**, masque compris. Les autres
politiques `scope="train"` et `method="percentile"` existant dans `modalities.py`
ne sont pas utilisées ici. Les FITS exportés restent avant normalisation.

Avant normalisation, le rendu uniquement utilise asinh P1–P99 par panneau ; les
colorbars portent les unités natives. Après normalisation, l’échelle est [0,1].
Le gris représente les pixels invalides dans les deux colonnes. Les figures sont
uniquement enregistrées puis fermées, jamais affichées dans le notebook.

La reprise compare le contenu des sources, la configuration, le code et les
versions des bibliothèques, puis vérifie les fichiers produits. Un résultat
modifié, incomplet ou en erreur est recalculé. Les écritures sont atomiques par
fichier et le manifeste est écrit en dernier. Ne pas lancer deux exécutions
concurrentes dans le même répertoire de sortie.


In [ ]:
rows = run_validation(
    records, FITS_ROOT, OUTPUT_DIR, config=config, cosmology=COSMOLOGY,
    cluster_ids=CLUSTER_IDS, overwrite=OVERWRITE,
    catalog_provenance={"url": CATALOG_URL, **fingerprint(CATALOG_PATH)},
)
summary = pd.DataFrame(rows)
print(f"Résultats : {OUTPUT_DIR}")


## Résumé et diagnostics

`has_*` décrit la présence du fichier source, même s’il est corrompu ; `status`
et `error_message` distinguent ces cas. La couverture Legacy globale est la
moyenne des bandes effectivement traitées ; le détail par bande et les absences
figurent dans les métadonnées. Un seuil bas produit un avertissement, sans exclusion.
`no_data` signifie qu’aucune modalité n’a pu être produite.


In [ ]:
if summary.empty:
    print("Aucun amas sélectionné.")
else:
    print(f"Amas examinés : {len(summary)} ; repris : {summary.resumed.sum()}")
    print(f"Fallback : {(summary.fov_method == 'angular_fallback').sum()}")
    display(summary.status.value_counts().rename("nombre"))
    display(summary[[c for c in summary if c.startswith("has_")]].sum().rename("fichiers présents"))
    numeric = ["fov_arcmin", "pixel_scale_arcsec"] + [c for c in summary if c.startswith("coverage_")]
    display(summary[numeric].apply(pd.to_numeric, errors="coerce").describe())
    print("Couvertures insuffisantes (aucune exclusion automatique) :")
    display(summary.loc[summary.low_coverage.fillna("").ne(""), ["cluster_id", "low_coverage"] + numeric])
    print("Erreurs / données indisponibles :")
    display(summary.loc[summary.status.ne("ok"), ["cluster_id", "status", "error_message"]])
